In [ ]:
# Strawberry Disease Detection using YOLOv8
# Complete Google Colab notebook
# Dataset must contain YOLO bounding-box labels (.txt).

!pip install ultralytics -q

import os, zipfile, shutil, yaml
import torch
from ultralytics import YOLO
from google.colab import files
from IPython.display import Image, display

print('Ultralytics YOLO installed successfully!')
print('GPU available:', torch.cuda.is_available())

In [ ]:
# Upload YOLO dataset ZIP
# Expected structure:
# strawberry_yolo/
#   images/train, images/val, images/test
#   labels/train, labels/val, labels/test

uploaded = files.upload()
zip_file = list(uploaded.keys())[0]
print('Uploaded:', zip_file)

In [ ]:
# Extract dataset
EXTRACT_DIR = '/content'
with zipfile.ZipFile(zip_file, 'r') as z:
    z.extractall(EXTRACT_DIR)
print('Dataset extracted!')

In [ ]:
# Find YOLO dataset directory automatically
def find_yolo_dataset(base='/content'):
    for root, dirs, fs in os.walk(base):
        if 'images' in dirs and 'labels' in dirs:
            return root
    return None

DATASET_DIR = find_yolo_dataset()
if DATASET_DIR is None:
    raise FileNotFoundError('Could not find a folder containing both images/ and labels/. Check your ZIP structure.')
print('Dataset directory:', DATASET_DIR)

In [ ]:
# Inspect folders
for folder in ['images/train','images/val','images/test','labels/train','labels/val','labels/test']:
    p = os.path.join(DATASET_DIR, folder)
    if os.path.exists(p):
        print(folder, ':', len(os.listdir(p)), 'files')
    else:
        print(folder, ': NOT FOUND')

In [ ]:
# Create data.yaml
# Change class names if your annotations use different classes.
# Here: 0 = Healthy, 1 = Leaf_Scorch

CLASS_NAMES = {0: 'Healthy', 1: 'Leaf_Scorch'}

data = {
    'path': DATASET_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': CLASS_NAMES
}

DATA_YAML = os.path.join(DATASET_DIR, 'data.yaml')
with open(DATA_YAML, 'w') as f:
    yaml.dump(data, f, sort_keys=False)

print(open(DATA_YAML).read())

In [ ]:
# Validate YOLO training labels

def check_labels(label_dir, n_classes):
    problems = []
    for fn in os.listdir(label_dir):
        if not fn.endswith('.txt'):
            continue
        path = os.path.join(label_dir, fn)
        with open(path, 'r') as f:
            for line_no, line in enumerate(f, 1):
                parts = line.strip().split()
                if len(parts) != 5:
                    problems.append(f'{fn}:{line_no} expected 5 values')
                    continue
                try:
                    cls, x, y, w, h = map(float, parts)
                    if int(cls) != cls or not 0 <= int(cls) < n_classes:
                        problems.append(f'{fn}:{line_no} invalid class id')
                    if not all(0 <= v <= 1 for v in [x,y,w,h]):
                        problems.append(f'{fn}:{line_no} coordinates must be 0-1')
                except Exception as e:
                    problems.append(f'{fn}:{line_no} {e}')
    return problems

train_labels = os.path.join(DATASET_DIR, 'labels/train')
if os.path.exists(train_labels):
    problems = check_labels(train_labels, len(CLASS_NAMES))
    print('Label check:', 'OK' if not problems else f'{len(problems)} problems found')
    for p in problems[:20]: print(p)
else:
    print('Training labels folder not found.')

In [ ]:
# Load pretrained YOLOv8 Nano
model = YOLO('yolov8n.pt')
print('YOLOv8n loaded!')

In [ ]:
# Train
results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    pretrained=True,
    project='/content/runs',
    name='strawberry_disease_yolov8',
    plots=True
)
print('Training complete!')

In [ ]:
# Display training results
RESULTS_DIR = '/content/runs/strawberry_disease_yolov8'
results_png = os.path.join(RESULTS_DIR, 'results.png')
if os.path.exists(results_png):
    display(Image(filename=results_png))
else:
    print('results.png not found:', RESULTS_DIR)

In [ ]:
# Validate model
metrics = model.val(data=DATA_YAML, imgsz=640)
try:
    print('mAP50:', metrics.box.map50)
    print('mAP50-95:', metrics.box.map)
except Exception:
    print(metrics)

In [ ]:
# Display confusion matrix
cm = os.path.join(RESULTS_DIR, 'confusion_matrix.png')
if os.path.exists(cm):
    display(Image(filename=cm))
else:
    print('Confusion matrix not found.')

In [ ]:
# Upload an unseen strawberry image
uploaded_test = files.upload()
TEST_IMAGE = list(uploaded_test.keys())[0]
print('Testing:', TEST_IMAGE)

In [ ]:
# Predict
prediction_results = model.predict(
    source=TEST_IMAGE,
    conf=0.25,
    imgsz=640,
    save=True,
    save_txt=True
)
result = prediction_results[0]
print('Prediction complete!')

In [ ]:
# Display prediction
result.show()

In [ ]:
# Print detection details
if result.boxes is None or len(result.boxes) == 0:
    print('No object/disease detected.')
else:
    for box in result.boxes:
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])
        coords = box.xyxy[0].tolist()
        print('Class:', model.names[class_id])
        print(f'Confidence: {confidence*100:.2f}%')
        print('Bounding Box:', coords)
        print('----------------------')

In [ ]:
# Save/download best model
BEST_MODEL = os.path.join(RESULTS_DIR, 'weights', 'best.pt')
print('Best model:', BEST_MODEL)
if os.path.exists(BEST_MODEL):
    files.download(BEST_MODEL)
else:
    print('best.pt not found.')

In [ ]:
# Optional: export trained model to ONNX for later application integration
onnx_model = model.export(format='onnx')
print('ONNX model:', onnx_model)